In [ ]:
# ========================
# 07_metrics_to_semantic_with_skeleton.ipynb
# 從六大指標數據反過來生成 LLM 語義對齊文字，並在旁邊附加動態骨架以便對照
# ========================
import pandas as pd
import json
import numpy as np
import os
from pathlib import Path
from openai import OpenAI
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML, display, clear_output

# OpenAI API Key 設定
OPENAI_API_KEY = "..."
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY.strip()
client = OpenAI()
OPENAI_MODEL = "gpt-4o-mini"

print(f"OpenAI 模型已設定為：{OPENAI_MODEL}")

OpenAI 模型已設定為：gpt-4o-mini


In [ ]:
# ========================
# Discord Multi-Agent Bridge Setup
# ========================
from dance_discord_utils import DiscordAgentBridge

DISCORD_WEBHOOK_URL = "YOUR_DISCORD_WEBHOOK_URL"
discord_bridge = DiscordAgentBridge(DISCORD_WEBHOOK_URL)

print("✅ Discord 即時串流橋接器準備就緒！")

✅ Discord 即時串流橋接器準備就緒！


In [3]:
# 定義分析資料夾路徑
folder = Path("C:/Users/AW'z/Downloads/ballet_Analysis_Results/data/streetdance-13_Analysis_Results/13-09/")

# 讀取指標資料與骨架資料
energy_df = pd.read_csv(folder / "energy.csv")
geometry_df = pd.read_csv(folder / "geometry.csv")
stability_df = pd.read_csv(folder / "stability.csv")
sync_df = pd.read_csv(folder / "synchronization.csv")
trans_df = pd.read_csv(folder / "transition.csv")
skeleton_df = pd.read_csv(folder / "13-09_skeleton.csv")

# 載入街舞文化資料庫 (Lookup Table)
cultural_lib_path = Path("C:/Users/AW'z/Downloads/ballet_Analysis_Results/street_dance_cultural_library.json")
with open(cultural_lib_path, "r", encoding="utf-8") as f:
    cultural_library = json.load(f)

print(f"✅ 指標與骨架資料載入成功！")
print(f"✅ 街舞文化資料庫載入成功！共 {len(cultural_library)} 筆項目")

✅ 指標與骨架資料載入成功！
✅ 街舞文化資料庫載入成功！共 157 筆項目


In [4]:
# 解析 skeleton.csv，將每幀的座標取出並轉換為 Numpy Array
num_frames = len(skeleton_df)
num_joints = 17
skel_data = np.zeros((num_frames, num_joints, 3))

for j in range(num_joints):
    col_str = skeleton_df[f'Joint_{j}']
    # 解析字串 'x, y, z' 到 float 陣列
    parsed = col_str.apply(lambda x: [float(v) for v in x.split(',')])
    skel_data[:, j, :] = np.vstack(parsed.values)

print(f"✅ 骨架資料解析完成！陣列形狀: {skel_data.shape} (Frames, Joints, XYZ)")

✅ 骨架資料解析完成！陣列形狀: (1933, 17, 3) (Frames, Joints, XYZ)


In [5]:
# 設定取樣間隔 (例如每 2 秒一個語義轉折點)
FPS = 30
INTERVAL_SEC = 2
INTERVAL_FRAMES = INTERVAL_SEC * FPS

total_frames = len(energy_df)
semantic_segments = []

for start_f in range(0, total_frames, INTERVAL_FRAMES):
    end_f = min(start_f + INTERVAL_FRAMES, total_frames)
    f_range = range(start_f, end_f)
    
    # 聚合這段時間的指標平均值
    seg_metrics = {
        'timestamp_sec': round(start_f / FPS, 2),
        'frame_start': start_f,
        'energy': energy_df.iloc[f_range]['energy'].mean(),
        'volume': geometry_df.iloc[f_range]['volume'].mean(),
        'curvature': geometry_df.iloc[f_range]['curvature'].mean(),
        'sway': stability_df.iloc[f_range]['sway'].mean(),
        'correlation': sync_df.iloc[f_range]['correlation'].mean(),
        'torque': trans_df.iloc[f_range]['torque'].mean(),
        'jerk': trans_df.iloc[f_range]['jerk'].mean()
    }
    semantic_segments.append(seg_metrics)

segments_df = pd.DataFrame(semantic_segments)
print(f"🔹 已切分為 {len(segments_df)} 個語義片段")

🔹 已切分為 33 個語義片段


In [6]:
# ========================
# Multi-Persona AI Logic (Sees & Says) - 3 Agents Version
# ========================

INTERVAL_SEC = 2.0

SYSTEM_PROMPT_SEES = """
你是一位街頭舞動分析專家。你的任務是根據提供的物理指標數據，對舞動進行客觀且技術性的描述。
請務必參考提供的「文化資料庫」，使用其中的 `term`（術語）與 `short_phrase`（短句）來描述動作。

【輸出格式】：
【AI sees】 [技術性描述]
【Keywords】 [本次描述中引用的術語，以逗號分隔]
"""

SYSTEM_PROMPTS_SAYS = {
    "Battle King": """
你是「Freestyle Battle 霸主」。請根據【AI sees】提供的動作觀察，即興噴出一句極具攻擊性與律動感的饒舌歌詞。
【創作準則】：
1. 語氣：Freestyle Battle 挑釁語氣，要狠、要有態度、要夠 Hype。
2. 內容：即興噴出一句極具攻擊性與律動感的饒舌歌詞。
3. 技巧：嚴禁死板重複術語！將能量與街頭態度轉化為饒舌意象與押韻。
4. 限制：嚴禁超過「一行」！維持單句爆發力。
【輸出格式】：
【AI says - Battle King】 [一行的 Freestyle Battle 歌詞]
""",
    "Old School": """
你是「街舞老砲兒」。你見證了街舞的黃金年代，講究的是靈魂（Soul）與根基（Foundation）。
【創作準則】：
1. 語氣：沈穩、前輩風範、講求技術與靈魂的傳承。
2. 內容：根據【AI sees】描述，給出一句點評，強調動作背後的技術點或文化底蘊。
3. 技巧：多提到 Foundation, Soul, Respect 等關鍵字。
4. 限制：嚴禁超過「一行」！
【輸出格式】：
【AI says - Old School】 [一句沈穩且具文化底蘊的點評]
""",
    "Shonen": """
你是「動漫熱血舞者」。你把每一場街舞都看作是燃燒靈魂的戰鬥，每一招都是必殺技。
【創作準則】：
1. 語氣：極度誇張、熱血、富有戲劇性。
2. 內容：將【AI sees】的動作轉化為動漫風格的「必殺技描述」。
3. 技巧：使用「靈壓」、「殘影」、「覺醒」、「最終形態」等熱血感十足的詞彙。
4. 限制：嚴禁超過「一行」！
【輸出格式】：
【AI says - Shonen】 [一句動漫感十足的熱血必殺技宣告]
"""
}

def safe_float(val):
    try:
        f_val = float(val)
        if np.isnan(f_val) or np.isinf(f_val):
            return 0.0
        return f_val
    except:
        return 0.0

def ai_sees(metrics, library):
    e = safe_float(metrics.get('energy', 0))
    v = safe_float(metrics.get('volume', 0))
    t = safe_float(metrics.get('torque', 0))
    j = safe_float(metrics.get('jerk', 0))
    
    lib_ref = "\n".join([f"- {item.get('term', 'N/A')}: {item.get('short_phrase', 'N/A')}" for item in library])
    
    prompt = f"""
    當前分析指標：
    - Energy: {e:.2f}, Volume: {v:.4f}
    - Torque: {t:.2f}, Jerk: {j:.2f}
    
    參考資料庫：
    {lib_ref}
    """
    
    try:
        response = client.chat.completions.create(
            model=OPENAI_MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT_SEES.strip()},
                {"role": "user", "content": prompt.strip()}
            ],
            temperature=0.3 
        )
        raw_output = response.choices[0].message.content.strip()
    except Exception as err:
        print(f"❌ AI Sees API 錯誤: {err}")
        return "【AI sees】 無法生成描述", "無"
    
    sees_part = "【AI sees】 無法解析描述"
    keywords_part = "無"
    for line in raw_output.split("\n"):
        if "【AI sees】" in line:
            sees_part = line
        elif "【Keywords】" in line:
            keywords_part = line.replace("【Keywords】", "").strip()
            
    return sees_part, keywords_part

def ai_says(sees_content, persona_name):
    """根據指定的角色人格生成回應"""
    try:
        prompt = SYSTEM_PROMPTS_SAYS.get(persona_name, "") 
        response = client.chat.completions.create(
            model=OPENAI_MODEL,
            messages=[
                {"role": "system", "content": prompt.strip()},
                {"role": "user", "content": sees_content.strip()}
            ],
            temperature=0.8 
        )
        return response.choices[0].message.content.strip()
    except Exception as err:
        print(f"❌ AI Says ({persona_name}) API 錯誤: {err}")
        return f"【AI says - {persona_name}】 訊號干擾中..."

print("✅ 三種人格 AI Agents 生成邏輯準備完成！")

✅ 三種人格 AI Agents 生成邏輯準備完成！


In [7]:
def create_skeleton_animation(skel_frames, fps=30):
    """將片段的 3D 骨架陣列繪製為動態對照的 HTML 影片"""
    fig = plt.figure(figsize=(4, 4))
    ax = fig.add_subplot(111, projection='3d')
    
    # 近似的 17 關節連線定義 (COCO/SMPL 風格)
    # 根據常見資料，若 0 是骨盆：
    bones = [
        (0, 1), (1, 2), (2, 3),        # 右腿
        (0, 4), (4, 5), (5, 6),        # 左腿
        (0, 7), (7, 8), (8, 9), (9, 10), # 軀幹與頭部
        (8, 11), (11, 12), (12, 13),   # 左手
        (8, 14), (14, 15), (15, 16)    # 右手
    ]
    
    lines = [ax.plot([], [], [], c='blue', lw=2)[0] for _ in bones]
    scat = ax.scatter([], [], [], c='red', s=20, alpha=0.5)
    
    # 設定適當的 3D 範圍
    ax.set_xlim([-1, 1])
    ax.set_ylim([-1, 1])
    ax.set_zlim([0, 2])
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    # 設定視角
    ax.view_init(elev=10, azim=0)
    plt.close(fig) # 隱藏靜態圖表
    
    def update(frame_idx):
        pts = skel_frames[frame_idx]
        scat._offsets3d = (pts[:,0], pts[:,1], pts[:,2])
        for line, bone in zip(lines, bones):
            p1, p2 = pts[bone[0]], pts[bone[1]]
            line.set_data([p1[0], p2[0]], [p1[1], p2[1]])
            line.set_3d_properties([p1[2], p2[2]])
        return lines + [scat]
    
    anim = animation.FuncAnimation(fig, update, frames=len(skel_frames), interval=1000/fps, blit=False)
    return HTML(anim.to_jshtml())

print("骨架動畫繪製邏輯準備完成！")

骨架動畫繪製邏輯準備完成！


In [8]:
import time, json
from IPython.display import clear_output, display

print("🚀 開始即時分析 (街舞模式: 2s/片段)... 各大門派高手即將集結！\n")

INTERVAL_FRAMES = int(INTERVAL_SEC * FPS)
results = []
personas = ["Battle King", "Old School", "Shonen"]

discord_bridge.send_status("🚀 開始全片即時分析及點評...")

for i, row in segments_df.iterrows():
    # 1. AI Sees
    discord_bridge.send_status(f"正在分析片段 {i+1}/{len(segments_df)} (T={row['timestamp_sec']}s)......")
    sees_content, keywords = ai_sees(row, cultural_library)
    
    # 2. Multi-Persona AI Says
    persona_responses = {}
    for p in personas:
        persona_responses[p] = ai_says(sees_content, p)
    
    res = {
        'timestamp': row['timestamp_sec'],
        'metrics': row.to_dict(),
        'sees': sees_content,
        'keywords': keywords,
        'says_battle_king': persona_responses["Battle King"],
        'says_old_school': persona_responses["Old School"],
        'says_shonen': persona_responses["Shonen"]
    }
    results.append(res)
    
    # 4. 即時推送到 Discord
    discord_bridge.push_segment(res)
    
    # 3. Immediate Display
    clear_output(wait=True)
    print("="*60)
    print(f"處理進度: {i+1}/{len(segments_df)} | 時間: {res['timestamp']} 秒")
    print("-" * 30)
    print(res['sees'])
    print("-" * 30)
    print(res['says_battle_king'])
    print(res['says_old_school'])
    print(res['says_shonen'])
    
    print("\n[AI Reasoning & Traceability]")
    print(f"- Linked Metrics: Energy={res['metrics']['energy']:.2f}, Torque={res['metrics']['torque']:.2f}")
    print(f"- Keywords from Library: {res['keywords']}")
    print(f"- Source: street_dance_cultural_library.json")
    
    print("\n[動態骨架對照]:")
    start_f = int(res['metrics']['frame_start'])
    end_f = int(min(start_f + INTERVAL_FRAMES, total_frames))
    display(create_skeleton_animation(skel_data[start_f:end_f]))
    
    # 4. Pacing Delay
    if i < len(segments_df) - 1:
        time.sleep(INTERVAL_SEC)

print("\n✨ 全部分析完成！正在匯出檔案...")

# 匯出檔案
with open('streetdance_multi_agent_results.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

# 僅匯出各 agent 的回應作為精華檔
ai_says_summary = []
for item in results:
    ai_says_summary.append({
        'timestamp': item['timestamp'],
        'Battle King': item['says_battle_king'],
        'Old School': item['says_old_school'],
        'Shonen': item['says_shonen']
    })

with open('streetdance_ai_says_multi_agents.json', 'w', encoding='utf-8') as f:
    json.dump(ai_says_summary, f, ensure_ascii=False, indent=2)

print("📁 已輸出：streetdance_multi_agent_results.json")
print("📁 已輸出：streetdance_ai_says_multi_agents.json")

discord_bridge.send_status("✨ 全部分析完成！檔案已自動儲存。")

處理進度: 33/33 | 時間: 64.0 秒
------------------------------
【AI sees】 根據當前的物理指標數據，舞者展現出相對較高的能量（1.13）和扭矩（2.64），這表明他們在動作中具有強烈的動力和旋轉能力。雖然音量較低（0.0441），但這可能意味著舞者在進行更細膩或內斂的動作。高達15976.28的急變（Jerk）指標顯示出他們在動作轉換時的瞬間加速度，這可能與Popping或Locking等風格中的快速肌肉收縮和停頓相呼應。整體來看，這些指標反映出舞者在技術上具備強大的控制力和表現力，能夠在舞蹈中創造出強烈的視覺衝擊。
------------------------------
【AI says - Battle King】 你這旋轉扭矩就像迷霧我穿透，舞台上瞬間加速，你根本跟不上我節奏！
【AI says - Old School】 雖然動作充滿力量與動態，但真正的靈魂來自於對根基的尊重與細膩的表現，才能讓每一次的瞬間加速變得更具意義。
【AI says - Shonen】「靈壓爆發！扭轉天地的『無限旋風舞』！瞬間加速，讓每一個動作如同風暴般撲面而來！」

[AI Reasoning & Traceability]
- Linked Metrics: Energy=1.13, Torque=2.64
- Keywords from Library: Energy, Torque, Jerk, Popping, Locking, Control, Visual Impact
- Source: street_dance_cultural_library.json

[動態骨架對照]:



✨ 全部分析完成！正在匯出檔案...
📁 已輸出：streetdance_multi_agent_results.json
📁 已輸出：streetdance_ai_says_multi_agents.json
